<font color="#117A65" size='6px'><b>Video Search Engine</b></font>


<font color="#117A65" size='4px'><b>The Pipeline Architecture</b></font>

The system operates in a high-dimensional vector search pipeline designed to bridge the gap between natural language queries and visual object tracks. It leverages **OpenCLIP** for multi-modal feature extraction and **FAISS** for industrial-scale similarity retrieval.



<font color="#D35400" size='4px'><b>Stage I: Feature Extraction & Embedding (Visual Memory)</b></font>
Before searching, the system must translate visual appearances into mathematical vectors.

**Multi-Modal Encoding:** It utilizes the <font color="#884EA0"><i>OpenCLIP (ViT-g/14)</i></font> model to analyze object crops generated by the tracking registry. 

**Track Distillation:** For every tracked object, the system selects the "best frame" (highest resolution/confidence) and generates a 1024-dimensional embedding that captures its semantic essence (e.g., color, shape, brand, or type).

<font color="#D35400" size='4px'><b>Stage II: Vector Database Indexing (FAISS)</b></font>
Once embeddings are created, they are organized for near-instantaneous retrieval.

**Index Construction:** The system builds a <font color="#884EA0"><i>FAISS (Facebook AI Similarity Search)</i></font> index. This converts raw embeddings into a searchable structure that supports L2 distance or Inner Product calculations.

**Metadata Mapping:** Every vector in the index is linked to a persistent metadata entry containing the object's <font color="#884EA0"><i>Trajectory</i></font>, <font color="#884EA0"><i>Timestamp</i></font>, and <font color="#884EA0"><i>Original Video Path</i></font>.

<font color="#D35400" size='4px'><b>Stage III: Multi-Modal Query Processing</b></font>
The engine allows users to search using different input types.

**Text Queries:** Natural language (e.g., "a person in a red shirt") is passed through the CLIP text tokenizer and encoder to create a query vector in the same latent space as the video objects.

**Image Queries:** If a user provides a reference image, the system generates a visual query vector, enabling "Find similar objects" functionality.

<font color="#D35400" size='4px'><b>Stage IV: Similarity Retrieval & Ranking</b></font>
The system performs a rapid "nearest neighbor" search in the vector space.

**Semantic Matching:** The engine calculates the similarity score between the query vector and every object in the database.

**Refinement & Filtering:** It applies a <font color="#CB4335"><i>Distance Threshold</i></font> to ensure only relevant results are returned, ranking them from highest to lowest confidence based on their cosine similarity.

<font color="#D35400" size='4px'><b>Stage V: Result Rendering & Visual Confirmation</b></font>
The final step translates mathematical results back into human-readable video segments.

**Temporal Localization:** The system uses the metadata to find the exact millisecond the object appeared in the video stream.

**Visual Verification:** It renders a dynamic preview of the result, drawing a <font color="#CB4335"><i>Bounding Box</i></font> around the searched object and displaying its motion trajectory to confirm the match was successful.

<font color="#2E86C1" size='6px'>Environment Initialization & Repository Setup</font>

In [ ]:
COLAB = False

In [ ]:
import os
import sys

if COLAB:
    # Clone Repository 
    if not os.path.exists("sam3-toolkit"):
        print("--- Cloning repository...")
        !git clone https://github.com/MrKGhasemi/sam3-toolkit.git
    else:
        print("--- Repository already exists.")

    # Auto-Detect Paths 
    repo_root = os.path.abspath("sam3-toolkit")
    sam3_install_path = None
    practical_path = None

    for root, dirs, files in os.walk(repo_root):
        if ("sam3" in dirs or "sam3_main" in root or root == repo_root):
            if sam3_install_path is None or len(root) < len(sam3_install_path):
                sam3_install_path = root

        if "video_search_engine.py" in files:
            practical_path = root

    if not sam3_install_path or not practical_path:
        raise FileNotFoundError(
            "     --- Critical paths (sam3 or practical) not found.")

    print(f"Found SAM 3 Library at: {sam3_install_path}")
    print(f"Found Project Code at: {practical_path}")

    # Install Dependencies 
    print("--- Installing Python packages...")
    !{sys.executable} -m pip install -q opencv-python matplotlib scikit-learn transformers spacy openai gdown mega.py huggingface_hub open_clip_torch
    !{sys.executable} -m pip install -q einops decord pycocotools

    # Install SAM 3 
    print("--- Installing SAM 3 Library...")
    !{sys.executable} -m pip install -e "{sam3_install_path}"

    # Configure Working Directory 
    os.chdir(practical_path)
    if practical_path not in sys.path:
        sys.path.insert(0, practical_path)
    print(f"--- Working directory set to: {os.getcwd()}")

    # Download Model Weights
    weights_dir = "weights"
    weights_path = os.path.join(weights_dir, "sam3.pt")
    os.makedirs(weights_dir, exist_ok=True)

    # If the model is Private/Gated, you MUST provide a token.
    # Get it here: https://huggingface.co/settings/tokens
    HF_REPO_ID = "facebook/sam3" 
    HF_FILENAME = "sam3.pt"               
    HF_TOKEN = "HF_TOKEN"  # Replace with your actual token or set to None

    if not os.path.exists(weights_path):
        print(f"--- Starting download...")

        try:
            from huggingface_hub import hf_hub_download
            print("--- Connecting to Hugging Face Hub...")
            downloaded_file = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                local_dir=weights_dir,
                token=HF_TOKEN,
                local_dir_use_symlinks=False  # Ensure we get the actual file, not a symlink
            )
            if os.path.basename(downloaded_file) != "sam3.pt":
                os.rename(downloaded_file, weights_path)

            print("--- Download attempt finished.")

        except Exception as e:
            print(f"      --- Error downloading: {e}")
            print("           Tip: If using Hugging Face private repo, ensure HF_TOKEN is set.")

    else:
        print("--- Weights file already exists.")

    # Verify File Integrity 
    if os.path.exists(weights_path):
        final_size = os.path.getsize(weights_path) / (1024 * 1024)
        print(f"--- Final File Size: {final_size:.2f} MB")
        if final_size < 2000:
            print(
                "      --- WARNING: File seems too small (<2GB). It might be corrupt or a placeholder.")
    else:
        print("      --- Error: File not found after download.")

    # Download SpaCy Model 
    print("--- Checking SpaCy model...")
    !{sys.executable} -m spacy download en_core_web_lg

    # Verify Import 
    print("\n--- Verifying imports...")
    repo_root = os.path.abspath("/sam3-toolkit")

    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
        print(f"--- Added '{repo_root}' to sys.path")

    practical_dir = os.path.join(repo_root, "practical")
    if practical_dir not in sys.path:
        sys.path.insert(0, practical_dir)
        print(f"--- Added '{practical_dir}' to sys.path")

    # Verify all dependencies
    print("\n--- Retrying imports...")
    try:
        import sam3
        print("--- 'sam3' library loaded!")
    except ImportError as e:
        print(f"      --- Failed to load sam3: {e}")
        print("       --- Attempting hard install fix...")
        !pip install "{repo_root}"
        import sam3

    try:
        import models
        print("--- 'models.py' loaded!")
    except ImportError as e:
        print(f"      --- Failed to load models: {e}")

    !pip uninstall numpy -y
    !{sys.executable} -m pip install "numpy==1.26.0" "faiss-cpu==1.8.0"    
    
    # Check GPU
    import torch
    print(f"--- GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"--- GPU Name: {torch.cuda.get_device_name(0)}")

    if os.path.exists(practical_dir):
        os.chdir(practical_dir)
        print(f"--- Current Directory: {os.getcwd()}")


In [ ]:
# !{sys.executable} -m pip install "numpy==1.26.0" "faiss-cpu==1.8.0"

---

In [ ]:
import os
import sys
root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root)

if os.path.basename(os.getcwd()) == "ipynb":
    os.chdir(root)

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

In [ ]:
import numpy
print(numpy.__version__) 
# process is Not capable with numpy 2.0.0 or higher

```Python
# From configs.py 
LLM_CAPTION_MODEL_NAME = "gpt-5-chat"
SAM3_CONF_VIDEO_SEARCH_ENGINE_THRESHOLD = 0.2
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs/"
os.makedirs(output_directory, exist_ok=True)


args = MockArgs(
    mode="llm",
    output_dir=output_directory,
    api_key="api_key",
    base_url="base_url"
)

---

In [ ]:
VIDEO_PATH = "videos/Traffic.mp4"

<font color="#117A65" size='6px'>Loading Core Modules & Dependencies</font>

<font color="#D68910" size='4px'>*build_video_index:*</font> Contains the main pipeline for identify, track, and create index of objects found.

<font color="#D68910" size='4px'>*search_video:*</font> pipiline for querying the objects entire the video.



In [ ]:
from video_search_engine import build_video_index, search_video

<font color="#117A65" size='6px'>Building the Visual Index</font>

This cell executes the Indexing Pipeline. By calling <font color="#117A65" size='4px'>build_video_index</font>, the system creates a searchable "visual memory" of the video. It translates every tracked object into a high-dimensional vector that can be queried using natural language.

The function orchestrates a multi-stage feature extraction pipeline:

<font color="#D35400" size='4px'><i>Temporal Object Tracking:</i></font> The engine first runs the complete tracking pipeline to identify every unique object in the video. It registers each instance into a <font color="#D35400"><i>Track Registry</i></font>, capturing its full trajectory and lifespan.

<font color="#D35400" size='4px'><i>Best-Frame Extraction:</i></font> For every unique track, the system selects the "best" visual crop—typically the frame where the object is largest or has the highest confidence score. This ensures the visual signature is based on the highest quality data available.

<font color="#D35400" size='4px'><i>CLIP Embedding Generation:</i></font> The selected crops are passed through the <font color="#884EA0"><b>OpenCLIP ViT-g/14</b></font> encoder. This model generates a 1024-dimensional feature vector (embedding) that encodes semantic concepts like color, brand, and type into mathematical coordinates.

<font color="#D35400" size='4px'><i>Vector Database Persistence:</i></font> These vectors are inserted into a FAISS (Facebook AI Similarity Search) index. Simultaneously, a metadata file (video_metadata.pkl) is saved, linking each vector ID back to its original video, timestamp, and class name.

```python
# Embedding logic from video_search_engine.py
# 1. Process crops with CLIP
image_input = self.preprocess(image).unsqueeze(0).to(self.device)
with torch.no_grad():
    image_features = self.model.encode_image(image_input)
    # L2 Normalization for Cosine Similarity
    image_features /= image_features.norm(dim=-1, keepdim=True)
    
# 2. Add to FAISS index
index.add(np.array(embeddings).astype('float32'))
```

In [ ]:
build_video_index(VIDEO_PATH, args)

In [ ]:
import faiss
import pickle
import os

base_path = "notebook_outputs/vector_db" 
index_file = os.path.join(base_path, "video_search.index")
meta_file = os.path.join(base_path, "video_metadata.pkl")

if not os.path.exists(index_file):
    index_file = "video_search.index"
    meta_file = "video_metadata.pkl"

print(f"Reading from:\n - {index_file}\n - {meta_file}\n")

print("--- [METADATA CONTENTS] ---")
if os.path.exists(meta_file):
    with open(meta_file, 'rb') as f:
        # Load the raw dictionary/list from the pickle
        metadata = pickle.load(f)
    
    print(f"Total Metadata Entries: {len(metadata)}")
    
print("\n--- [FAISS INDEX CONTENTS] ---")
if os.path.exists(index_file):
    # Load the index
    index = faiss.read_index(index_file)
    
    print(f"Index Type: {type(index)}")
    print(f"Total Vectors stored (ntotal): {index.ntotal}")
    print(f"Vector Dimensions (d): {index.d}")
    
    # EXTRACTING VECTORS
    try:
        vector_0 = index.reconstruct(0)
        print(f"\nRaw Vector for ID 0 (First 5 values): {vector_0[:5]}")
        print(f"Vector Shape: {vector_0.shape}")
    except RuntimeError:
        print("Note: This specific index type does not support direct vector reconstruction.")
else:
    print("❌ Index file not found.")

<font color="#8E44AD" size='6px'>Video Search: Semantic Query Retrieval</font>

This cell executes the Retrieval Pipeline. By calling <font color="#8E44AD" size='4px'>search_video("bmw")</font>, the system bridges the gap between your text query and the pre-computed visual database to find the most relevant video segments.

The search process follows a high-speed similarity pipeline:

<font color="#8E44AD" size='4px'><i>Zero-Shot Query Encoding:</i></font> The text string (e.g., "bmw") is tokenized and passed through the CLIP text encoder. This transforms the word into a query vector in the same latent space as the video embeddings, enabling direct mathematical comparison.

<font color="#8E44AD" size='4px'><i>Similarity Ranking:</i></font> The system performs a Nearest Neighbor search in the FAISS index. It calculates the distance (Cosine Similarity) between your query vector and every object in the database, returning the <font color="#8E44AD"><i>top_k</i></font> most similar matches.

<font color="#8E44AD" size='4px'><i>Metadata Retrieval & Localization:</i></font> For each result, the engine looks up the associated metadata. It identifies exactly which frame the object appeared in and its location (Bounding Box), allowing the system to verify the result visually.

<font color="#8E44AD" size='4px'><i>Result Visualization:</i></font> If <font color="#8E44AD"><i>save_video</i></font> is enabled, the system uses <font color="#8E44AD"><i>render_track_video</i></font> to cut and render a 2-3 second preview for each match, complete with a green tracking box and the query label for easy identification.

```python
# Search logic from video_search_engine.py
# 1. Encode query text
text_token = self.tokenizer([query_text]).to(self.device)
with torch.no_grad():
    text_features = self.model.encode_text(text_token)
    text_features /= text_features.norm(dim=-1, keepdim=True)

# 2. Search FAISS index
distances, indices = index.search(query_vec, top_k)

# 3. Retrieve and render results
for dist, idx in zip(distances[0], indices[0]):
    meta = metadata_registry[idx]
    # Localization and visualization logic...
```    

In [ ]:
search_video("bmw", top_k=5, save_video=False)

In [ ]:
import pickle
import os
from config import configs

# Load Metadata
meta_path = os.path.join(configs.VECTOR_DB_PATH, configs.METADATA_FILE)
with open(meta_path, 'rb') as f:
    data = pickle.load(f)

# Search for BMW entries
bmw_tracks = [d for d in data if "bmw" in d['class_name'].lower()]
all_cars = [d for d in data if "car" in d['class_name'].lower()]

print(f"Total Tracks: {len(data)}")
print(f"BMW Tracks found: {len(bmw_tracks)}")
print(f"Total 'Car' Tracks: {len(all_cars)}")

if len(bmw_tracks) == 0:
    print("\n❌ ISSUE FOUND: SAM 3 did not track the BMW!")
    print("   Reason: It might have been filtered out by the 'Area < 200' check")
    print("   or the 'Score < Threshold' check in Phase 2.")
else:
    print("\n✅ BMW is in the index. The issue is ranking.")
    for t in bmw_tracks:
        print(f"   - ID: {t['track_id']} (Frames: {len(t['trajectory'])})")

In [ ]:
search_video("porsche grey car", top_k=1, save_video=True)

In [ ]:
search_video("motor bike", top_k=2, save_video=True)

In [ ]:
search_video("red car", top_k=2, save_video=True)

In [ ]:
search_video("speed limit 40 sign", top_k=2, save_video=True)